In [ ]:
# Base pipeline configuration.
%reload_ext autoreload
%autoreload 2
%cd ~/erc-src/cuneiform-ocr-sign-alignment-worktree
%env PATH=$HOME/.local/bin:$PATH

import os

import sign_alignment.pipeline as pp
from sign_alignment.data_source import LocalDataSource, PrototypeSource
from sign_alignment.detector import ModelConfig, TabletImageDetector
from sign_alignment.dift_align import DiftAlignmentConfig, DiftRuntime
from sign_alignment.visualizer import ColorConfig

ANNOTATIONS_DIR = os.path.expanduser(
    "~/erc-work-data/data-of-cuneiform-ocr-data/filtered_annotations"
)
CONFIG_FILE = "configs/detr.py"
CHECKPOINT_FILE = os.path.expanduser(
    "~/erc-work-data/retrained_models/detr-173/epoch_1000.pth"
)
DIFT_CHECKPOINT = os.path.expanduser("~/erc-src/ProtoSnap/weights/SD_with_prompt")
CANONICAL_FEATURE_DIR = os.path.expanduser(
    "~/erc-work-data/signs_alignment_data/precompute_feautures"
)
SCORE_THRESHOLD = 0.0
OUTPUT_DIR = "alignment_results"
SAMPLE_INDEX = 0
CROP_INDEX = 3


In [ ]:
# Initialize reusable detector, DIFT runtime, and base context.
model_config = ModelConfig(
    config_file=CONFIG_FILE,
    checkpoint_file=CHECKPOINT_FILE,
    device="auto",
)
if "tablet_detector" not in globals() or getattr(tablet_detector, "model", None) is None:
    tablet_detector = TabletImageDetector(
        default_score_threshold=SCORE_THRESHOLD,
        model_config=model_config,
        keep_crops=True,
        is_crop_itself=False,
    )
else:
    print("Reusing existing tablet_detector instance.")

dift = DiftRuntime(
    checkpoint=DIFT_CHECKPOINT,
    feature_dir=CANONICAL_FEATURE_DIR,
    config=DiftAlignmentConfig(affine_probe_padding_ratio=0.1),
)
context = pp.CropContext(
    tablet_detector=tablet_detector,
    local_source=LocalDataSource(ANNOTATIONS_DIR),
    color_config=ColorConfig,
    output_dir=OUTPUT_DIR,
    img_idx=CROP_INDEX,
    dift=dift,
    sign_source=PrototypeSource(),
)
runner = pp.Runner(context, pp.VisOptions(info=True, display=True, save=True))


In [ ]:
# Load one sample and detect signs in the selected crop.
runner.choose_sample(SAMPLE_INDEX)
runner.run([
    pp.Step("Load data", pp.load_data, pp.vis_loaded_data),
    pp.Step("Detect signs", pp.detect_signs, pp.vis_detections),
])


In [ ]:
# Run the complete base coarse-alignment flow.
runner.run([
    pp.Step("Transform GT to crop", pp.transform_gt_to_crop, pp.vis_crop_ground_truth),
    pp.Step("Detection statistics", lambda _: None, pp.vis_detection_statistics),
    pp.Step("Create box sets", pp.create_box_sets, pp.vis_box_sets),
    pp.Step("Detect rows", pp.detect_rows, pp.vis_detected_rows_info),
    pp.Step("Match rows", pp.match_rows, pp.vis_row_matches),
    pp.Step("Visualize rows", lambda _: None, pp.vis_detection_rows),
    pp.Step("Match signs", pp.match_signs_in_rows, pp.vis_sign_matches),
    pp.Step("Align text rows", pp.align_text_rows, pp.vis_aligned_rows),
])


In [ ]:
# Compare coarse alignment with unchanged detection geometry.
runner.run([
    pp.Step(
        "Result without optimization",
        pp.create_result_without_optimization,
        pp.vis_result_without_optimization,
    ),
    pp.Step("Build sign match info", pp.build_sign_match_info, pp.vis_sign_match_info),
    pp.Step("Offset analysis", lambda _: None, pp.vis_offset_analysis),
])


In [ ]:
# Release detector VRAM, initialize source signs, and run PSR.
runner.run([
    pp.Step("Unload detector", pp.unload_detector),
    pp.Step("Setup source signs", pp.setup_source_signs),
    pp.Step("Create PSR optimizer", pp.create_psr_optimizer, pp.vis_psr_optimizer),
    pp.Step("Optimize until DIFT probe", pp.optimize_psr_until_dift_probe, pp.vis_optimization),
    pp.Step("Source sign overlay", pp.create_source_sign_overlay, pp.vis_source_sign_overlay),
    pp.Step("DIFT affine probe", pp.run_dift_affine_probe, pp.vis_dift_affine_probe),
    pp.Step("Finish PSR optimization", pp.optimize_psr_after_dift_probe, pp.vis_optimization),
])


In [ ]:
# Inspect the final optimization results.
runner.run([
    pp.Step("Loss history", lambda _: None, pp.vis_loss_history),
    pp.Step("Results comparison", lambda _: None, pp.vis_results_comparison),
    pp.Step("Parameter changes", lambda _: None, pp.vis_parameter_changes),
])
